# Tag 11 · UE 2 · Arbeitsauftrag 1: Fahrradzubehör in Python

Die Aufgabe verlangt zuerst Denken, nicht Code: Welche Fragen kann die Tabelle beantworten, welche Darstellung passt, welche Erkenntnis ist zu erwarten? Dieses Notebook setzt die acht Fragen aus `11_UE2_Loesung_Arbeitsauftrag1_Fahrradzubehoer.md` mit **matplotlib** um. Alle Beträge in Tsd. €, Datenstand Rechnungen 2025 ohne Retouren.

Gestaltungsregeln: eine Aussage je Grafik (Titel nennt die Erkenntnis), eine Achse, feste Farbe je Produktlinie, nur ausgewählte Werte beschriftet, sortiert nach Größe.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Farben: feste Zuordnung je Produktlinie (kategorisch, Reihenfolge blau, orange, aqua, gelb)
FARBE = {"Einsteiger": "#2a78d6", "Standard": "#eb6834", "Profi": "#1baf7a", "Ersatzteile": "#eda100"}
BLAU, ROT, GRAU = "#2a78d6", "#e34948", "#898781"
TINTE, TINTE2, GITTER = "#0b0b0b", "#52514e", "#e1e0d9"

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTE2, "text.color": TINTE,
    "xtick.color": TINTE2, "ytick.color": TINTE2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GITTER, "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titleweight": "bold", "axes.titlesize": 12,
    "axes.titlelocation": "left", "font.size": 10, "legend.frameon": False,
})
def fmt(x):  # deutsches Zahlenformat
    return f"{x:,.0f}".replace(",", ".")

## 1. Daten und Plausibilitätsprüfung

In [ ]:
linien = ["Einsteiger", "Standard", "Profi", "Ersatzteile"]
df = pd.DataFrame(
    [[410, 620, 180, 240], [380, 710, 540, 260], [520, 480, 90, 310],
     [290, 330, 60, 150], [350, 560, 220, 200]],
    index=["Nord", "Süd", "West", "Ost", "Mitte"], columns=linien)
vorjahr = pd.Series([1390, 1640, 1520, 810, 1250], index=df.index, name="Vorjahr")
haendler = pd.Series([64, 81, 72, 38, 55], index=df.index)
aussendienst = pd.Series([2, 3, 2, 1, 2], index=df.index)
marge = pd.Series({"Einsteiger": .22, "Standard": .33, "Profi": .41, "Ersatzteile": .55})

# Gesamt-Spalte der Folie nachrechnen
gesamt_folie = pd.Series([1450, 1890, 1400, 830, 1330], index=df.index)
assert (df.sum(axis=1) == gesamt_folie).all(), "Zeilensummen stimmen nicht mit der Folie überein"

tab = pd.DataFrame({"Gesamt": df.sum(axis=1), "Vorjahr": vorjahr})
tab["Δ abs"] = tab["Gesamt"] - tab["Vorjahr"]
tab["Δ %"] = (tab["Gesamt"] / tab["Vorjahr"] - 1) * 100
print("Gesamt 2025:", tab["Gesamt"].sum(), "| Vorjahr:", tab["Vorjahr"].sum(),
      f"| {tab['Gesamt'].sum() / tab['Vorjahr'].sum() * 100 - 100:+.1f} %")
tab.round(1)

Die Zeilensummen stimmen mit der Spalte „Gesamt“ der Folie überein. Das Gesamtplus von 4,4 % ist die Ausgangsgröße. Was dahinter steckt, zeigen die Regionen.

## 2. Frage 1: Wie hat sich der Umsatz je Region gegenüber dem Vorjahr verändert?

**Form:** Hantel-Diagramm (Vorjahr → 2025). Es zeigt den Vorher-Nachher-Vergleich je Region und die Richtung auf einen Blick. Wachstum blau, Rückgang rot, zusätzlich mit Vorzeichen beschriftet, damit die Farbe nie allein trägt.

In [ ]:
t = tab.sort_values("Δ %")
fig, ax = plt.subplots(figsize=(8, 3.6))
for i, (reg, z) in enumerate(t.iterrows()):
    farbe = BLAU if z["Δ abs"] >= 0 else ROT
    ax.plot([z["Vorjahr"], z["Gesamt"]], [i, i], color=farbe, lw=2.5, solid_capstyle="round", zorder=2)
    ax.scatter(z["Vorjahr"], i, s=70, facecolor="#fcfcfb", edgecolor=farbe, lw=2, zorder=3)
    ax.scatter(z["Gesamt"], i, s=70, color=farbe, zorder=3)
    rechts = z["Δ abs"] >= 0
    ax.text(max(z["Vorjahr"], z["Gesamt"]) + 40, i, f"{z['Δ abs']:+.0f} ({z['Δ %']:+.1f} %)".replace(".", ","),
            va="center", fontsize=9.5, color=TINTE2)
ax.set_yticks(range(len(t)), t.index)
ax.set_xlim(600, 2250); ax.grid(axis="y", visible=False)
ax.set_xlabel("Umsatz in Tsd. €   (○ Vorjahr, ● 2025)")
ax.set_title("Süd wächst deutlich, West ist als einzige Region im Minus", pad=12)
plt.tight_layout(); plt.show()

**Erkenntnis:** Süd +250 (+15,2 %) trägt fast das ganze Gesamtplus. West verliert 120 (−7,9 %). Die Streuung wird erst auf Regionsebene sichtbar.

## 3. Frage 2: Wie viel trägt jede Region zum Gesamtumsatz bei?

**Form:** Horizontale Balken, nach Größe sortiert. Ein Kreisdiagramm wäre bei fünf ähnlich großen Teilen schlechter ablesbar.

In [ ]:
anteil = (tab["Gesamt"] / tab["Gesamt"].sum() * 100).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 3.2))
b = ax.barh(anteil.index, anteil.values, height=0.55, color=BLAU)
for y, (reg, v) in enumerate(anteil.items()):
    ax.text(v + 0.4, y, f"{v:.1f} % ({fmt(tab.loc[reg, 'Gesamt'])})".replace(".", ","), va="center", fontsize=9.5, color=TINTE2)
ax.set_xlim(0, 34); ax.grid(axis="y", visible=False)
ax.set_xlabel("Anteil am Gesamtumsatz 2025 in %")
ax.set_title("Süd und Nord erbringen fast die Hälfte des Umsatzes", pad=10)
plt.tight_layout(); plt.show()

**Erkenntnis:** Süd (27 %) und Nord (21 %) sind die größten Regionen, Ost (12 %) die kleinste.

## 4. Frage 3: Wie setzt sich der Umsatz je Region aus den Produktlinien zusammen?

**Form:** Gestapelte Balken, links absolut (Größe), rechts als 100 %-Variante (Mix). Segmente tragen 2 px Abstand zum Hintergrund, Beschriftung nur, wo sie hineinpasst.

In [ ]:
reihenfolge = tab.sort_values("Gesamt", ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, anteilig in zip(axes, (False, True)):
    d = df.loc[reihenfolge]
    if anteilig:
        d = d.div(d.sum(axis=1), axis=0) * 100
    links = np.zeros(len(d))
    for l in linien:
        ax.barh(d.index, d[l], left=links, height=0.6, color=FARBE[l], edgecolor="#fcfcfb", linewidth=2, label=l)
        for y, (v, x0) in enumerate(zip(d[l], links)):
            if v >= (11 if anteilig else 160):
                ax.text(x0 + v / 2, y, f"{v:.0f}" + (" %" if anteilig else ""), ha="center", va="center",
                        fontsize=8.5, color="#0b0b0b" if l == "Ersatzteile" else "white")
        links += d[l].values
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Anteil in %" if anteilig else "Umsatz in Tsd. €")
    ax.set_title("Mix der Linien" if anteilig else "Umsatz je Region", pad=8)
axes[0].invert_yaxis()
axes[0].legend(ncol=4, loc="upper center", bbox_to_anchor=(1.05, -0.2))
fig.suptitle("Süd ist profilastig, West einsteigerlastig", x=0.07, ha="left", fontweight="bold", fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

**Erkenntnis:** In Süd macht Profi 29 % des Umsatzes aus, in West nur 6 %. Die Regionen haben ganz unterschiedliche Sortimentsprofile. Ein Gesamtvergleich nach Umsatz verdeckt das.

## 5. Frage 4: Welche Produktlinie bringt wie viel Umsatz und wie viel Rohertrag?

Der Rohertrag je Linie ist Umsatz × Marge. Da die Marge nur je Linie vorliegt, gilt sie hier für alle Regionen gleich. Das ist eine **Näherung**.

**Form:** Gruppierte Balken mit einer gemeinsamen Achse (Anteil in %), je Linie Umsatzanteil und Rohertragsanteil nebeneinander. Keine zweite Skala.

In [ ]:
umsatz_l = df.sum()
rohertrag_l = umsatz_l * marge
vergleich = pd.DataFrame({
    "Umsatz": umsatz_l, "Marge": marge, "Rohertrag": rohertrag_l,
    "Anteil Umsatz %": umsatz_l / umsatz_l.sum() * 100,
    "Anteil Rohertrag %": rohertrag_l / rohertrag_l.sum() * 100})
print(f"Gesamt-Rohertrag: {rohertrag_l.sum():.0f}  (Ø-Marge {rohertrag_l.sum() / umsatz_l.sum() * 100:.1f} %)")
vergleich.round(1)

In [ ]:
x = np.arange(len(linien)); w = 0.36
fig, ax = plt.subplots(figsize=(8.5, 3.9))
b1 = ax.bar(x - w/2, vergleich["Anteil Umsatz %"], w - 0.04, color="#9ec5f4", label="Anteil am Umsatz")
b2 = ax.bar(x + w/2, vergleich["Anteil Rohertrag %"], w - 0.04, color=BLAU, label="Anteil am Rohertrag")
for bars in (b1, b2):
    for r in bars:
        ax.text(r.get_x() + r.get_width()/2, r.get_height() + 0.7, f"{r.get_height():.1f} %".replace(".", ","),
                ha="center", fontsize=9, color=TINTE2)
ax.set_xticks(x, linien); ax.grid(axis="x", visible=False); ax.set_ylim(0, 46)
ax.set_ylabel("Anteil in %")
ax.set_title("Ersatzteile und Profi bringen mehr Ertrag als Umsatz, Einsteiger weniger", pad=10)
ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

**Erkenntnis:** Standard ist der Motor (39 % Umsatz, 37 % Rohertrag). Ersatzteile haben 17 % Umsatz, aber 27 % Rohertrag. Einsteiger bringen 28 % Umsatz, aber nur 18 % Rohertrag.

## 6. Frage 5: Lohnt es sich, die Profi-Linie einzustellen?

Zwei Blicke: Was würde wegfallen (Rohertrag je Linie in Tsd. €) und wo hängt Profi dran (Profi-Umsatz je Region)?

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 1]})

r = rohertrag_l.sort_values()
a1.barh(r.index, r.values, height=0.55, color=[FARBE[l] if l == "Profi" else "#c3c2b7" for l in r.index])
for y, (l, v) in enumerate(r.items()):
    a1.text(v + 15, y, fmt(v), va="center", fontsize=9.5, color=TINTE2)
a1.set_xlim(0, 1050); a1.grid(axis="y", visible=False)
a1.set_xlabel("Rohertrag in Tsd. €"); a1.set_title("Profi bringt mehr Rohertrag als Einsteiger", pad=8)

p = df["Profi"].sort_values()
a2.barh(p.index, p.values, height=0.55, color=FARBE["Profi"])
for y, (reg, v) in enumerate(p.items()):
    a2.text(v + 8, y, f"{v}  ({v / df['Profi'].sum() * 100:.0f} %)", va="center", fontsize=9.5, color=TINTE2)
a2.set_xlim(0, 680); a2.grid(axis="y", visible=False)
a2.set_xlabel("Profi-Umsatz in Tsd. €  (Klammer: Anteil an Profi gesamt)")
a2.set_title("Die Hälfte des Profi-Umsatzes kommt aus Süd", pad=8)
plt.tight_layout(); plt.show()

print(f"Ohne Profi: Rohertrag −{rohertrag_l['Profi'] / rohertrag_l.sum() * 100:.1f} %, "
      f"Umsatz −{umsatz_l['Profi'] / umsatz_l.sum() * 100:.1f} %")
print(f"Süd ohne Profi: {df.loc['Süd'].sum() - df.loc['Süd', 'Profi']} statt {df.loc['Süd'].sum()} (Vorjahr {vorjahr['Süd']})")

**Erkenntnis:** Profi ist klein (15,8 % des Umsatzes), aber ertragsstark (18,6 % des Rohertrags) und regional konzentriert. Ohne Profi läge Süd bei 1.350, also unter dem Vorjahr (1.640). Eine Einstellung wäre mit diesen Daten nicht zu empfehlen. Offen bleiben Kosten und Verbundkäufe, siehe unten.

## 7. Frage 6: Wo ist der Vertrieb produktiv?

**Form:** Zwei einfache Balkendiagramme nebeneinander (Umsatz je Händler, Umsatz je Außendienstmitarbeiter). Beide haben eine eigene Einheit, deshalb zwei Grafiken und keine Doppelachse.

In [ ]:
prod = pd.DataFrame({"je Händler": tab["Gesamt"] / haendler, "je Außendienst": tab["Gesamt"] / aussendienst})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, spalte, einheit in zip(axes, prod.columns, ("Tsd. € je Händler", "Tsd. € je Außendienstmitarbeiter")):
    s = prod[spalte].sort_values()
    farben = [ROT if reg == "West" and spalte == "je Händler" else BLAU for reg in s.index]
    ax.barh(s.index, s.values, height=0.55, color=farben)
    for y, v in enumerate(s.values):
        ax.text(v + s.max() * 0.015, y, f"{v:.1f}".replace(".", ",") if spalte == "je Händler" else f"{v:.0f}",
                va="center", fontsize=9.5, color=TINTE2)
    ax.set_xlim(0, s.max() * 1.18); ax.grid(axis="y", visible=False); ax.set_xlabel(einheit)
axes[0].set_title("Je Händler liegen die Regionen eng, West fällt ab", pad=8)
axes[1].set_title("Ost führt, aber mit nur einer Stelle", pad=8)
plt.tight_layout(); plt.show()

**Erkenntnis:** Je Händler liegen die Regionen zwischen 19 und 24 Tsd. €, West ist der Ausreißer nach unten. Ost hat den höchsten Umsatz je Außendienst, weil nur eine Stelle besetzt ist, die seit Juli leer ist. Das ist ein Risiko für 2026.

*Optional als Streudiagramm (Händleranzahl gegen Umsatz).* Mit nur fünf Punkten zeigt es Orientierung, keine statistisch belastbare Korrelation.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(haendler, tab["Gesamt"], s=90, color=BLAU, zorder=3)
for reg in df.index:
    ax.annotate(reg, (haendler[reg], tab.loc[reg, "Gesamt"]), xytext=(7, 5), textcoords="offset points", fontsize=9.5)
ax.set_xlabel("Anzahl Händler"); ax.set_ylabel("Umsatz 2025 in Tsd. €")
ax.set_title("Mehr Händler, mehr Umsatz? Nur fünf Punkte, nur Orientierung", pad=10)
plt.tight_layout(); plt.show()

## 8. Frage 7: Wie viel Rohertrag bringt jede Region?

Wie bei Frage 4 gilt die Linien-Marge für alle Regionen (Näherung).

In [ ]:
roh_reg = df.mul(marge, axis=1)
roh_reg["Gesamt"] = roh_reg.sum(axis=1)
roh_reg = roh_reg.sort_values("Gesamt")
fig, ax = plt.subplots(figsize=(8.5, 3.6))
links = np.zeros(len(roh_reg))
for l in linien:
    ax.barh(roh_reg.index, roh_reg[l], left=links, height=0.58, color=FARBE[l], edgecolor="#fcfcfb", linewidth=2, label=l)
    links += roh_reg[l].values
for y, (reg, v) in enumerate(roh_reg["Gesamt"].items()):
    ax.text(v + 8, y, f"{v:.0f}  (Umsatz {fmt(tab.loc[reg, 'Gesamt'])})", va="center", fontsize=9.5, color=TINTE2)
ax.set_xlim(0, 880); ax.grid(axis="y", visible=False)
ax.set_xlabel("Rohertrag in Tsd. € (geschätzt mit Linien-Marge)")
ax.set_title("Süd liefert mit 682 den meisten Rohertrag, Ost mit 280 den wenigsten", pad=10)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=9)
plt.tight_layout(); plt.show()

**Erkenntnis:** Süd bringt 682 von 2.405 (28 %). West (Umsatz 1.400, Rohertrag 480) liegt beim Ertrag trotz fast gleichen Umsatzes hinter Nord (1.450, 501), weil es viele Einsteiger (22 % Marge) und kaum Profi verkauft. Der Mix bestimmt den Ertrag, nicht nur der Umsatz.

## 9. Frage 8: Wie ist der Produktmix je Region im Vergleich?

**Form:** Heatmap Region × Produktlinie, eine Farbe (hell → dunkel), Zahlen in den Zellen, sodass auch der Ausdruck in Graustufen lesbar bleibt.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.8))
cmap = LinearSegmentedColormap.from_list("blau", ["#e8f1fc", "#86b6ef", "#2a78d6", "#0d366b"])
ax.grid(False)
im = ax.imshow(df.values, cmap=cmap, aspect="auto")
ax.set_xticks(range(4), linien); ax.set_yticks(range(5), df.index)
ax.xaxis.tick_top(); ax.tick_params(length=0)
for s in ax.spines.values(): s.set_visible(False)
for i in range(df.shape[0]):
    for j in range(df.shape[1]):
        v = df.iat[i, j]
        ax.text(j, i, v, ha="center", va="center", fontsize=11,
                color="white" if v > df.values.max() * 0.55 else TINTE)
ax.set_title("Hotspots in Süd (Standard 710, Profi 540), Profi in Ost und West kalt", pad=26, loc="left", fontsize=11)
plt.tight_layout(); plt.show()

**Erkenntnis:** Südliche Profi- und Standardverkäufe stechen heraus. Ost und West sind bei Profi die kältesten Zellen (60 und 90).

## 10. Was die Daten nicht hergeben

| Fehlende Daten | Wofür sie nötig wären |
|---|---|
| Vorjahr je Produktlinie | Trend der Profi-Linie, Wachstumstreiber in Süd |
| Monats- oder Quartalswerte | Zeitpunkt des Einbruchs in West (März), Effekt des neuen Großhändlers in Nord (Mai), Saison |
| Ost-Umsatz über den Nord-Großhändler | Fairer Regionsvergleich (Ost wirkt zu klein, Nord zu groß) |
| Kosten je Linie (Vertrieb, Lager, Marketing) | Gewinn statt Rohertrag, Einstellungsentscheidung |
| Verbundkäufe | Cross-Selling-Effekt bei Profi |
| Retouren | Nettoumsatz |
| Plan- oder Zielwerte | Einordnung, ob +4,4 % gut ist |
| Marge je Region | Ersatz für die Annahme einer einheitlichen Marge je Linie |

Bewusst nicht verwendet: Katalog-Redesign und Firmensitz seit 1987 haben keinen Bezug zu den Fragen. Der Werksverkauf in Mitte kann den Händlervergleich verzerren.